# 00 ? Contrato de batches e smoke de Chronos-2

## Registro do piloto

- Gerador tsfmxai: seed 20261112; 6 canais por sistema; 8 exemplos; contexto 32; horizonte 16; grammar com 1?2 operadores e lags 1?4.
- Checkpoint: `amazon/chronos-2` @ `29ec3766d36d6f73f0696f85560a422f50e8498c`.
- Ambiente: Windows, Python 3.11.9, PyTorch 2.12.0+cpu; CUDA indispon?vel.
- Escopo: adapta??o batch e um forward/backward real para confirmar loss/gradientes; sem optimizer step, checkpoint treinado ou avalia??o cient?fica.

## Reproduction instructions

Use Python 3.11 from the repository root. Install the project dependencies with `python -m pip install -e ".[learning,dev,release]"`, then execute this notebook with:

`python -m nbconvert --to notebook --execute --inplace experiments/generator_model_evaluation/00_batch_contract_smoke.ipynb`

The Chronos-2 checkpoint used by this experiment is pinned to `amazon/chronos-2@29ec3766d36d6f73f0696f85560a422f50e8498c`; the skeleton teacher, where used, is pinned to `mateuswiteck/tsfmxai-canonicalization-encoder-decoder` variant `transformer_skeleton` at `2fcdcc88b41125bcfd92953c0bb8a668a603d167`. The notebook records its own seeds, settings, observed result, and current limitations.
The Chronos Python implementation is read from `experiments_local/chronos-forecasting`; recreate that ignored checkout at commit `10afa9ebe016e514f9d7dc1aa873f66af57e116b` and install it with `python -m pip install -e experiments_local/chronos-forecasting`.


In [1]:
from pathlib import Path
import sys
import torch
torch.manual_seed(20261112)
ROOT=Path.cwd()
while not (ROOT/'src/tsfmxai').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
if not (ROOT/'src/tsfmxai').exists(): raise RuntimeError('repository root not found')
sys.path.insert(0,str(ROOT/'src'))
from tsfmxai.datasets import (TrajectoryConfig, TrainingFormat, generate_training_batches)
from tsfmxai.generators.combinatorial_expression import CombinatorialTreeConfig
from tsfmxai.generators.initialization import HistorySpec
from tsfmxai.generators.multiseries import MultiseriesConfig, NoiseConfig
from tsfmxai.datasets.chronos2_adapter import adapt_tsfmxai_batch
print('torch',torch.__version__,'cuda',torch.cuda.is_available())

torch 2.12.0+cpu cuda False


In [2]:
config=MultiseriesConfig(
 ast_config=CombinatorialTreeConfig(minimum_operators=1,maximum_operators=2,minimum_lag=1,maximum_lag=4,max_degree=2),
 channel_count_probs={6:1.0}, dependency_probs={'random':1.0},
 treatment_probs={'original':1.0}, max_external_sources=2,
 noise=NoiseConfig(std_range=(0.02,0.05)))
stream=generate_training_batches(
 config=config,num_systems=16,trajectories_per_system=1,seed=20261112,
 trajectory=TrajectoryConfig(128,HistorySpec('uniform','uniform'),max_initialization_retries=3),
 training=TrainingFormat(context_length=32,prediction_length=16,stride=16,batch_size=8,dtype='float32'))
batch=next(stream)
chronos_batch=adapt_tsfmxai_batch(batch, layout='batch_time_channel', output_patch_size=16)
rows=chronos_batch['context'].shape[0]
assert rows == 48
assert chronos_batch['context'].shape[1:]==(32,)
assert chronos_batch['future_target'].shape[1:]==(16,)
assert chronos_batch['equation_targets'].shape[0]==rows
assert chronos_batch['noise_std'].shape==(rows,)
assert torch.bincount(chronos_batch['group_ids']).tolist()==[6]*8
print('tsfmxai context',batch.context.shape,'future',batch.future.shape)
print('Chronos rows',chronos_batch['context'].shape, 'groups',torch.unique(chronos_batch['group_ids']).numel())
print('Equation target rows',chronos_batch['equation_targets'].shape,'noise targets',chronos_batch['noise_std'].shape)
print('systems generated',stream.stats.attempted_systems,'accepted trajectories',stream.stats.accepted_trajectories)
assert chronos_batch['num_output_patches']==1
assert torch.isnan(chronos_batch['future_covariates']).all()

tsfmxai context (8, 32, 6) future (8, 16, 6)
Chronos rows torch.Size([48, 32]) groups 8
Equation target rows torch.Size([48, 7]) noise targets torch.Size([48])
systems generated 6 accepted trajectories 2


In [3]:
# Real Chronos checkpoint integration smoke for one tsfmxai-generated batch.
chronos_src=ROOT/'experiments_local/chronos-forecasting/src'
sys.path.insert(0,str(chronos_src))
from chronos.chronos2.pipeline import Chronos2Pipeline
from time import perf_counter
MODEL_ID='amazon/chronos-2'; MODEL_REVISION='29ec3766d36d6f73f0696f85560a422f50e8498c'
pipeline=Chronos2Pipeline.from_pretrained(MODEL_ID,revision=MODEL_REVISION,device_map='cpu')
model=pipeline.model.train(); model.zero_grad(set_to_none=True)
t0=perf_counter()
out=model(context=chronos_batch['context'],group_ids=chronos_batch['group_ids'],
          future_covariates=chronos_batch['future_covariates'],
          num_output_patches=chronos_batch['num_output_patches'],
          future_target=chronos_batch['future_target'])
assert torch.isfinite(out.loss) and out.quantile_preds.shape==(48,21,16)
out.loss.backward()
gradients=sum(p.grad is not None for p in model.parameters())
assert gradients>0
print({'generator':'tsfmxai','checkpoint_revision':MODEL_REVISION,'loss':float(out.loss.detach()),
       'prediction_shape':tuple(out.quantile_preds.shape),'grad_parameter_tensors':gradients,
       'cpu_forward_backward_s':round(perf_counter()-t0,3),'optimizer_update':False})
model.zero_grad(set_to_none=True); model.eval()

C:\Users\mateu\repos\tsfmxai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'generator': 'tsfmxai', 'checkpoint_revision': '29ec3766d36d6f73f0696f85560a422f50e8498c', 'loss': 12.42272663116455, 'prediction_shape': (48, 21, 16), 'grad_parameter_tensors': 170, 'cpu_forward_backward_s': 1.343, 'optimizer_update': False}


Chronos2Model(
  (shared): Embedding(2, 768)
  (input_patch_embedding): ResidualBlock(
    (dropout): Dropout(p=0.1, inplace=False)
    (hidden_layer): Linear(in_features=48, out_features=3072, bias=True)
    (act): ReLU()
    (output_layer): Linear(in_features=3072, out_features=768, bias=True)
    (residual_layer): Linear(in_features=48, out_features=768, bias=True)
  )
  (patch): Patch()
  (instance_norm): InstanceNorm()
  (encoder): Chronos2Encoder(
    (block): ModuleList(
      (0-11): 12 x Chronos2EncoderBlock(
        (layer): ModuleList(
          (0): TimeSelfAttention(
            (self_attention): MHA(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (rope_embed): Chronos2RotaryEmbedding()
            )
            (layer

## Resultado da verifica??o

Smoke test executado com seis canais fixos por exemplo: oito exemplos multivariados viram 48 linhas Chronos-2, organizadas em oito grupos de seis. Targets de equa??o e ru?do foram alinhados ?s mesmas linhas. O treino exige ainda checkpoint/revis?o Chronos-2 e GPU, indispon?veis neste ambiente.

Distribui??es com n?mero vari?vel de canais precisam ser agrupadas por contagem de canais antes de formar batches retangulares.